# NGO Automation — Enrichment Pipeline (Colab)

This notebook runs the full enrichment pipeline on Google Colab:

1. **Upload** your `OSU.xlsx` file + project code (as ZIP)
2. **Enrich** each professor from 13 public sources
3. **Chunk** merged text via LLM
4. **Upload** embeddings to Pinecone
5. **Sync** structured profiles to MongoDB

---

### Prerequisites
- Your `NGO-Automation` project zipped as `NGO-Automation.zip`
- `OSU.xlsx` Excel file with faculty profile URLs
- API keys: `OPENAI_API_KEY`, `PINECONE_API_KEY`, `MONGODB_URI`

## Step 0: Install Dependencies

In [ ]:
!pip install -q \
    python-dotenv httpx pinecone pymongo openai tqdm \
    openpyxl pandas beautifulsoup4 pdfplumber PyPDF2 python-docx \
    sentence-transformers tiktoken semantic-text-splitter \
    langchain langchain-text-splitters nltk \
    scholarly duckduckgo-search PyMuPDF \
    pydantic aiofiles nest-asyncio requests numpy boto3 psutil \
    playwright

# Install playwright browsers (needed for web scraping in full pipeline)
!playwright install chromium

print("\n✅ All dependencies installed!")

## Step 1: Upload Project Code

**Option A (recommended):** Upload `NGO-Automation.zip` — a zip of your entire project folder.

To create the zip on your local machine:
```
# From the parent directory of NGO-Automation:
# PowerShell:
Compress-Archive -Path .\NGO-Automation\* -DestinationPath .\NGO-Automation.zip

# Or use 7-Zip / WinRAR to zip the NGO-Automation folder contents
```

**Option B:** Mount Google Drive if you've already synced the project there.

In [ ]:
import os, sys

# === Choose your upload method ===
USE_GOOGLE_DRIVE = False  # Set True to mount Google Drive instead of uploading zip
GDRIVE_PROJECT_PATH = "/content/drive/MyDrive/NGO-Automation"  # Only used if USE_GOOGLE_DRIVE=True

PROJECT_DIR = "/content/NGO-Automation"

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = GDRIVE_PROJECT_PATH
    print(f"✅ Using Google Drive project at: {PROJECT_DIR}")
else:
    from google.colab import files
    import zipfile

    print("Upload your NGO-Automation.zip file:")
    uploaded = files.upload()

    zip_name = list(uploaded.keys())[0]
    print(f"\nExtracting {zip_name}...")

    os.makedirs(PROJECT_DIR, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall(PROJECT_DIR)

    print(f"✅ Project extracted to {PROJECT_DIR}")

# Detect nested zip (e.g. NGO-Automation/NGO-Automation/...)
nested = os.path.join(PROJECT_DIR, "NGO-Automation")
if os.path.isdir(nested) and os.path.exists(os.path.join(nested, "run_enrichment_pipeline.py")):
    PROJECT_DIR = nested
    print(f"⚠️ Detected nested folder — using: {PROJECT_DIR}")

# Also check if files are one level deeper with any folder name
if not os.path.exists(os.path.join(PROJECT_DIR, "run_enrichment_pipeline.py")):
    subdirs = [d for d in os.listdir(PROJECT_DIR) if os.path.isdir(os.path.join(PROJECT_DIR, d))]
    for sd in subdirs:
        candidate = os.path.join(PROJECT_DIR, sd)
        if os.path.exists(os.path.join(candidate, "run_enrichment_pipeline.py")):
            PROJECT_DIR = candidate
            print(f"⚠️ Found project inside subfolder — using: {PROJECT_DIR}")
            break

os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

# Verify
assert os.path.exists("run_enrichment_pipeline.py"), (
    f"❌ run_enrichment_pipeline.py not found in {PROJECT_DIR}.\n"
    f"Contents: {os.listdir(PROJECT_DIR)}\n"
    "Make sure your zip contains the project files (not a wrapper folder)."
)

print(f"\n✅ Working directory: {os.getcwd()}")
print(f"✅ sys.path[0]: {sys.path[0]}")
print(f"✅ Key files found: {[f for f in ['run_enrichment_pipeline.py', 'full_automation_pipeline.py', 'enrichment'] if os.path.exists(f)}]")

## Step 2: Set API Keys

Enter your API keys below. These are needed for:
- **OPENAI_API_KEY** — LLM chunking + summaries + embeddings
- **PINECONE_API_KEY** — Vector database storage
- **MONGODB_URI** — MongoDB Atlas connection string

In [ ]:
import os
from getpass import getpass

# --- Option 1: Use Colab Secrets (recommended) ---
# Go to the key icon in the left sidebar, add your secrets there,
# then set USE_COLAB_SECRETS = True
USE_COLAB_SECRETS = False

if USE_COLAB_SECRETS:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["PINECONE_API_KEY"] = userdata.get("PINECONE_API_KEY")
    os.environ["MONGODB_URI"] = userdata.get("MONGODB_URI")
    # Optional:
    try:
        os.environ["YOUTUBE_API_KEY"] = userdata.get("YOUTUBE_API_KEY")
    except Exception:
        pass
else:
    # --- Option 2: Manual input ---
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ")
    os.environ["PINECONE_API_KEY"] = getpass("Enter PINECONE_API_KEY: ")
    os.environ["MONGODB_URI"] = getpass("Enter MONGODB_URI: ")
    yt_key = input("Enter YOUTUBE_API_KEY (press Enter to skip): ").strip()
    if yt_key:
        os.environ["YOUTUBE_API_KEY"] = yt_key

# Write .env file so the pipeline scripts can load it
with open(".env", "w") as f:
    f.write(f'OPENAI_API_KEY={os.environ["OPENAI_API_KEY"]}\n')
    f.write(f'PINECONE_API_KEY={os.environ["PINECONE_API_KEY"]}\n')
    f.write(f'MONGODB_URI={os.environ["MONGODB_URI"]}\n')
    if os.environ.get("YOUTUBE_API_KEY"):
        f.write(f'YOUTUBE_API_KEY={os.environ["YOUTUBE_API_KEY"]}\n')

print("\n✅ API Keys configured:")
print(f"  OPENAI_API_KEY:   {'set' if os.environ.get('OPENAI_API_KEY') else 'MISSING'}")
print(f"  PINECONE_API_KEY: {'set' if os.environ.get('PINECONE_API_KEY') else 'MISSING'}")
print(f"  MONGODB_URI:      {'set' if os.environ.get('MONGODB_URI') else 'MISSING'}")
print(f"  YOUTUBE_API_KEY:  {'set' if os.environ.get('YOUTUBE_API_KEY') else 'skipped (optional)'}")

## Step 3: Upload OSU Excel File

Upload your `OSU.xlsx` file containing the faculty profile URLs.

In [ ]:
from google.colab import files
import shutil

print("Upload your OSU.xlsx file:")
uploaded = files.upload()

excel_filename = list(uploaded.keys())[0]
excel_path = os.path.join(os.getcwd(), "OSU.xlsx")

if excel_filename != "OSU.xlsx":
    shutil.move(excel_filename, excel_path)
    print(f"Renamed {excel_filename} → OSU.xlsx")

import pandas as pd
df = pd.read_excel(excel_path)
print(f"\n✅ Excel loaded: {len(df)} rows, columns: {list(df.columns)}")
df.head(3)

## Step 4: Choose What to Run

If you already have profiles from a previous local run (included in your zip),
you can **skip straight to enrichment**.

Otherwise, run the full pipeline first to create profiles from the Excel file.

In [ ]:
from pathlib import Path

profiles_dir = Path("output/osu_faculty_run/profiles")
existing_profiles = 0
if profiles_dir.exists():
    existing_profiles = len([d for d in profiles_dir.iterdir() if d.is_dir()])

print(f"Existing profiles found: {existing_profiles}")

if existing_profiles > 0:
    print(f"\n✅ You have {existing_profiles} profiles already.")
    print("You can skip to Step 5 (Enrichment) directly.")
    print("Or run Step 4b below to add more profiles from the Excel file.")
else:
    print("\n⚠️ No profiles found. Run Step 4b to create them from your Excel file.")

### Step 4b: Run Full Pipeline (Excel → Profiles) — Only if needed

This scrapes each profile URL from the Excel file, cleans text, chunks it,
and creates profile directories. Skip this if you already uploaded profiles.

In [ ]:
# ⚠️ Only run this cell if you DON'T already have profiles
# Adjust START_FROM and LIMIT to control which profiles to process

START_FROM = 1        # 1-based profile number to start from
LIMIT = None          # Set to e.g. 50 to process a batch, or None for all
SKIP_PINECONE = True  # Skip Pinecone during profile creation (enrichment handles it)
SKIP_MONGODB = True   # Skip MongoDB during profile creation (enrichment handles it)

# --- Ensure path is set (safe to re-run) ---
import os, sys
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import nest_asyncio
nest_asyncio.apply()

os.environ.setdefault("INTENT_GATING_ENABLED", "0")
os.environ.setdefault("STRICT_SOURCE_POLICY", "0")

from dotenv import load_dotenv
load_dotenv()

from full_automation_pipeline import run_full_pipeline

output_dir = "output/osu_faculty_run"
chunking_output_dir = f"{output_dir}/chunked_profiles"

await run_full_pipeline(
    excel_path="OSU.xlsx",
    output_dir=output_dir,
    chunking_output_dir=chunking_output_dir,
    use_llm_chunking=True,
    llm_provider="openai",
    llm_model="gpt-4o-mini",
    limit=LIMIT,
    start_from=START_FROM - 1,
    skip_pinecone=SKIP_PINECONE,
    skip_mongodb=SKIP_MONGODB,
)

## Step 5: Run Enrichment Pipeline

This is the main pipeline — enriches each professor from 13 public sources,
then chunks, uploads to Pinecone, and syncs to MongoDB.

Adjust the parameters below to control the run.

In [ ]:
# ═══════════════════════════════════════════════════════════
# ENRICHMENT PIPELINE CONFIGURATION
# ═══════════════════════════════════════════════════════════

# -- Range control --
START = 0             # 0-based index to start from
LIMIT = None          # Max profiles to enrich (None = all remaining)
NAME = None           # Enrich single professor by name, e.g. "Amber Bruney"
FORCE = False         # Force re-enrich even if already done

# -- Pipeline steps --
SKIP_PINECONE = False # Skip Pinecone upload
SKIP_MONGODB = False  # Skip MongoDB sync
SKIP_CHUNKING = False # Skip chunking + Pinecone + MongoDB (collect data only)

# -- Source control --
DISABLED_SOURCES = None   # e.g. "google_scholar" to disable slow sources
ENABLED_SOURCES = None    # e.g. "semantic_scholar,openalex" to run only these

# -- Performance --
MAX_CONCURRENT = 4    # Max concurrent API requests per professor
MIN_CONFIDENCE = 0.0  # Skip chunk/upload below this confidence

print("Configuration set. Run the next cell to start.")

In [ ]:
# --- Ensure path is set (safe to re-run) ---
import os, sys
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
os.chdir(PROJECT_DIR)

import nest_asyncio
nest_asyncio.apply()

import asyncio
import json
import logging
import time
from pathlib import Path

from dotenv import load_dotenv
load_dotenv()

from enrichment.orchestrator import (
    EnrichmentOrchestrator,
    load_professor_queries_from_profiles,
    ALL_COLLECTORS,
)
from enrichment.base_collector import ProfessorQuery

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("enrichment_pipeline")

profiles_dir = Path("output/osu_faculty_run/profiles")
output_dir = Path("output/osu_faculty_run")

if not profiles_dir.exists():
    raise FileNotFoundError(f"Profiles directory not found: {profiles_dir}. Run Step 4b first.")

# Parse source filters
enabled = ENABLED_SOURCES.split(",") if ENABLED_SOURCES else None
disabled = DISABLED_SOURCES.split(",") if DISABLED_SOURCES else None

# Load professor queries
print(f"\n[Setup] Loading professor profiles from: {profiles_dir}")
if NAME:
    all_queries = load_professor_queries_from_profiles(str(profiles_dir))
    name_lower = NAME.lower()
    queries = [q for q in all_queries if name_lower in q.name.lower()]
    if not queries:
        raise ValueError(f"No professor found matching name: {NAME}")
    print(f"[Setup] ✅ Found {len(queries)} professors matching '{NAME}'")
else:
    queries = load_professor_queries_from_profiles(
        str(profiles_dir),
        limit=LIMIT,
        start_from=START,
        filter_no_enrichment=not FORCE,
    )
    print(f"[Setup] ✅ Loaded {len(queries)} profiles (start={START}, limit={LIMIT or 'all'}, force={FORCE})")

if not queries:
    print("[Setup] ⚠️ No profiles to enrich. Set FORCE=True to re-enrich already-processed profiles.")
else:
    print(f"\n[Preview] First 10 professors to enrich:")
    for i, q in enumerate(queries[:10], 1):
        print(f"  {i}. {q.name} (dept={q.department or '?'})")
    if len(queries) > 10:
        print(f"  ... and {len(queries) - 10} more")

In [ ]:
# ═══════════════════════════════════════════════════════════
# RUN THE ENRICHMENT PIPELINE
# ═══════════════════════════════════════════════════════════

async def run_enrichment():
    from enrichment.incremental_sync import incremental_sync_professor

    # Determine pipeline mode
    mode_parts = ["enrich"]
    if not SKIP_CHUNKING:
        mode_parts.append("chunk")
    if not SKIP_PINECONE and not SKIP_CHUNKING:
        mode_parts.append("pinecone")
    if not SKIP_MONGODB and not SKIP_CHUNKING:
        mode_parts.append("mongodb")

    print(f"\n{'='*80}")
    print(f"ENRICHMENT PIPELINE")
    print(f"  Profiles → Enrich (13 sources) → Chunk → Pinecone → MongoDB")
    print(f"{'='*80}")
    print(f"[Pipeline] Profiles to process: {len(queries)}")
    print(f"[Pipeline] Pipeline mode: {' → '.join(mode_parts)}")
    print(f"[Pipeline] Max concurrent: {MAX_CONCURRENT}")
    print(f"[Pipeline] Env: OPENAI_API_KEY={'✅' if os.getenv('OPENAI_API_KEY') else '❌'}, "
          f"PINECONE_API_KEY={'✅' if os.getenv('PINECONE_API_KEY') else '❌'}, "
          f"MONGODB_URI={'✅' if os.getenv('MONGODB_URI') else '❌'}")

    # Initialize orchestrator
    orchestrator = EnrichmentOrchestrator(
        output_dir=str(output_dir),
        enabled_sources=enabled,
        disabled_sources=disabled,
        max_concurrent=MAX_CONCURRENT,
    )

    start_time = time.perf_counter()
    total = len(queries)
    success_count = 0
    fail_count = 0
    pinecone_total = 0
    mongo_total = 0
    chunks_total = 0
    source_stats = {}

    try:
        for i, query in enumerate(queries, 1):
            prof_start = time.perf_counter()
            print(f"\n{'━'*80}")
            print(f"[{i}/{total}] Processing: {query.name}")
            if query.department:
                print(f"  Department: {query.department}")
            print(f"{'━'*80}")

            try:
                # Step 1: Collect enrichment data
                print(f"  [Step 1/5] Collecting enrichment data...")
                results = await orchestrator.enrich_professor(query)

                # Step 2: Save enrichment files
                print(f"  [Step 2/5] Saving enrichment files...")
                enrichment_path = orchestrator.save_enrichment(query, results)
                text_path = orchestrator.save_enrichment_text(query, results)

                successful_sources = [n for n, r in results.items() if r.success]
                failed_sources = [n for n, r in results.items() if not r.success]
                text_saved = text_path is not None

                # Read confidence
                confidence = 0.0
                try:
                    enr_data = json.loads(enrichment_path.read_text(encoding="utf-8"))
                    confidence = enr_data.get("confidence", {}).get("overall_confidence", 0.0)
                except Exception:
                    pass

                # Track per-source stats
                for s in successful_sources:
                    source_stats[s] = source_stats.get(s, 0) + 1
                for s in failed_sources:
                    source_stats.setdefault(s, 0)

                chunks_count = 0
                pinecone_uploaded = 0
                mongodb_synced = False

                # Steps 3-5: Chunk → Pinecone → MongoDB
                if successful_sources and text_saved and not SKIP_CHUNKING:
                    if MIN_CONFIDENCE <= 0 or confidence >= MIN_CONFIDENCE:
                        profile_dir = output_dir / "profiles" / query.profile_id
                        chunking_output_dir = output_dir / "chunked_profiles"

                        sync_result = incremental_sync_professor(
                            profile_dir=profile_dir,
                            chunking_output_dir=chunking_output_dir,
                            professor_name=query.name,
                            skip_pinecone=SKIP_PINECONE,
                            skip_mongodb=SKIP_MONGODB,
                        )

                        chunks_count = sync_result.get("chunks_count", 0)
                        pinecone_uploaded = sync_result.get("pinecone_uploaded", 0)
                        mongodb_synced = sync_result.get("mongodb_synced", False)

                chunks_total += chunks_count
                pinecone_total += pinecone_uploaded
                if mongodb_synced:
                    mongo_total += 1

                elapsed = time.perf_counter() - prof_start
                n_success = len(successful_sources)

                if n_success > 0:
                    print(f"\n  ✅ [{i}/{total}] DONE: {query.name}")
                    success_count += 1
                else:
                    print(f"\n  ❌ [{i}/{total}] NO DATA: {query.name}")
                    fail_count += 1

                status = f"{n_success}/{n_success + len(failed_sources)} sources"
                if chunks_count:
                    status += f" | {chunks_count} chunks"
                if pinecone_uploaded:
                    status += f" | {pinecone_uploaded} vectors→Pinecone"
                if mongodb_synced:
                    status += " | MongoDB ✅"
                print(f"     {status} | confidence={confidence:.2f} | {elapsed:.1f}s")

            except Exception as e:
                fail_count += 1
                elapsed = time.perf_counter() - prof_start
                print(f"\n  ❌ [{i}/{total}] EXCEPTION: {query.name} — {e} ({elapsed:.1f}s)")
                import traceback
                traceback.print_exc()

            # Delay between professors
            if i < total:
                await asyncio.sleep(5.0)

    finally:
        await orchestrator.close()

    elapsed_total = time.perf_counter() - start_time
    success_rate = (success_count / max(total, 1)) * 100

    print(f"\n{'='*80}")
    print(f"[SUMMARY] Enrichment Pipeline Complete!")
    print(f"{'='*80}")
    print(f"  Professors processed:  {total}")
    print(f"  Successful (≥1 src):   {success_count}")
    print(f"  Failed (0 sources):    {fail_count}")
    print(f"  Success rate:          {success_rate:.1f}%")
    print(f"  Chunks generated:      {chunks_total:,}")
    print(f"  Pinecone vectors:      {pinecone_total:,}")
    print(f"  MongoDB synced:        {mongo_total}")
    print(f"  Total time:            {elapsed_total:.1f}s ({elapsed_total/max(total,1):.1f}s avg)")
    print(f"{'─'*80}")
    print(f"  Per-source success rates:")
    for source_name in sorted(source_stats.keys()):
        count = source_stats[source_name]
        pct = count / max(total, 1) * 100
        bar = "█" * int(pct / 5) + "░" * (20 - int(pct / 5))
        print(f"    {source_name:<22} {count:>4}/{total} ({pct:5.1f}%) {bar}")
    print(f"{'='*80}")

await run_enrichment()

## Step 6: Check Results

In [ ]:
from pathlib import Path
import json

profiles_dir = Path("output/osu_faculty_run/profiles")

total = 0
enriched = 0
chunked = 0
sources_found = {}

for pdir in sorted(profiles_dir.iterdir()):
    if not pdir.is_dir():
        continue
    total += 1

    enr_json = pdir / "enrichment.json"
    if enr_json.exists():
        enriched += 1
        try:
            data = json.loads(enr_json.read_text(encoding="utf-8"))
            for src in data.get("summary", {}).get("successful_source_names", []):
                sources_found[src] = sources_found.get(src, 0) + 1
        except Exception:
            pass

    chunks_dir = Path("output/osu_faculty_run/chunked_profiles") / pdir.name
    if (chunks_dir / "chunks.json").exists():
        chunked += 1

print(f"{'='*60}")
print(f"PIPELINE RESULTS")
print(f"{'='*60}")
print(f"  Total profiles:    {total}")
print(f"  Enriched:          {enriched}")
print(f"  Chunked:           {chunked}")
print(f"  Remaining:         {total - enriched}")
print(f"\n  Source coverage:")
for src in sorted(sources_found.keys()):
    print(f"    {src:<22} {sources_found[src]:>4}/{enriched}")
print(f"{'='*60}")

## Step 7: Check MongoDB Results

In [ ]:
import os
from pymongo import MongoClient
from config.mongodb_utils import create_mongo_client, resolve_mongo_db_name

mongodb_uri = os.getenv("MONGODB_URI")
client = create_mongo_client(mongodb_uri)
db_name = resolve_mongo_db_name(mongodb_uri)
db = client[db_name]

scholars_count = db.scholars.count_documents({})
print(f"\n{'='*60}")
print(f"MONGODB STATUS")
print(f"{'='*60}")
print(f"  Database:    {db_name}")
print(f"  Collection:  scholars")
print(f"  Documents:   {scholars_count}")

# Show a few sample names
if scholars_count > 0:
    print(f"\n  Sample scholars:")
    for doc in db.scholars.find({}, {"name.full": 1, "about.current_position": 1}).limit(5):
        name = doc.get("name", {}).get("full", "?")
        pos = doc.get("about", {}).get("current_position", "")
        print(f"    - {name}: {pos}")

print(f"{'='*60}")
client.close()

## Step 8: Download Results (Optional)

Download the enrichment output as a zip file to your local machine.

In [ ]:
import shutil
from google.colab import files

output_zip = "/content/enrichment_output"
print("Zipping output directory...")
shutil.make_archive(output_zip, "zip", "output/osu_faculty_run")
print(f"✅ Created {output_zip}.zip")

files.download(f"{output_zip}.zip")